# EMT 3-phase Composite State-Space Extraction

This notebook validates state-space extraction for supported EMT three-phase composite components.

Two electrically equivalent models are compared:

1. **Composite model**: `NetworkInjection`, `PiLine`, and `RXLoad`.
2. **Explicit model**: the same network assembled from the corresponding voltage source, resistor, inductor, and capacitor subcomponents.

For each model, the notebook extracts the discrete-time state matrix and checks:

- the expected number and ordering of extraction states,
- equality of the complete discrete-time state matrices,
- equality of the discrete-time eigenvalues.

The `RXLoad` is validated for inductive, capacitive, and purely resistive reactive-power configurations. No plots are generated.

In [ ]:
import numpy as np
import dpsimpy

emt = dpsimpy.emt
ph3 = emt.ph3

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Domain = dpsimpy.Domain
Solver = dpsimpy.Solver
PhaseType = dpsimpy.PhaseType
StateSpaceModalAnalysis = dpsimpy.StateSpaceModalAnalysis

## Parameters

In [ ]:
frequency = 50.0
omega = 2.0 * np.pi * frequency

time_step = 50e-6
final_time = 2.0 * time_step

nominal_voltage = 20e3

line_resistance_value = 0.05
line_inductance_value = 0.1
line_capacitance_value = 0.1e-6
line_conductance_value = 1e-6

load_active_power_value = 100e3

# Positive Q creates an inductor, negative Q creates a capacitor,
# and zero Q creates no reactive subcomponent.
load_cases = {
    "inductive": 50e3,
    "capacitive": -50e3,
    "resistive": 0.0,
}

matrix_tolerance = 1e-12
eigenvalue_tolerance = 1e-10

## Helper functions

In [ ]:
def three_phase_parameter(value):
    return np.array(
        dpsimpy.Math.single_phase_parameter_to_three_phase(value),
        dtype=float,
        copy=True,
    )


def three_phase_power(value):
    return np.array(
        dpsimpy.Math.single_phase_power_to_three_phase(value),
        dtype=float,
        copy=True,
    )


line_resistance = three_phase_parameter(line_resistance_value)
line_inductance = three_phase_parameter(line_inductance_value)
line_capacitance = three_phase_parameter(line_capacitance_value)
line_conductance = three_phase_parameter(line_conductance_value)

load_active_power = three_phase_power(load_active_power_value)


def configure_and_run(system, simulation_name):
    simulation = Simulation(simulation_name)
    simulation.set_system(system)
    simulation.set_domain(Domain.EMT)
    simulation.set_solver(Solver.MNA)
    simulation.set_time_step(time_step)
    simulation.set_final_time(final_time)
    simulation.do_state_space_extraction(True)
    simulation.run()
    return simulation


def get_extraction_results(simulation):
    extractor = simulation.get_state_space_extractor()

    modal_analysis = StateSpaceModalAnalysis(extractor)
    modal_analysis.update()

    return {
        "state_count": extractor.get_state_count(),
        "state_names": list(modal_analysis.get_state_names()),
        "Ad": np.array(
            extractor.get_discrete_state_matrix(),
            dtype=float,
            copy=True,
        ),
        "z": np.array(
            modal_analysis.get_discrete_eigenvalues(),
            dtype=np.complex128,
            copy=True,
        ).reshape(-1),
    }


def sort_complex(values):
    return np.array(
        sorted(
            np.asarray(values).reshape(-1),
            key=lambda value: (
                round(float(value.imag), 12),
                round(float(value.real), 12),
            ),
        ),
        dtype=np.complex128,
    )


def expected_state_names(reactive_power):
    names = [
        "Line_ind_a",
        "Line_ind_b",
        "Line_ind_c",
        "Line_cap0_a",
        "Line_cap0_b",
        "Line_cap0_c",
        "Line_cap1_a",
        "Line_cap1_b",
        "Line_cap1_c",
    ]

    if reactive_power > 0.0:
        names.extend(["Load_ind_a", "Load_ind_b", "Load_ind_c"])
    elif reactive_power < 0.0:
        names.extend(["Load_cap_a", "Load_cap_b", "Load_cap_c"])

    return names

## Composite model

In [ ]:
def run_composite_model(reactive_power, case_name):
    source_node = emt.SimNode("n1", PhaseType.ABC)
    load_node = emt.SimNode("n2", PhaseType.ABC)

    source = ph3.NetworkInjection("Slack")
    source.set_parameters(
        dpsimpy.Math.single_phase_variable_to_three_phase(nominal_voltage),
        frequency,
    )

    line = ph3.PiLine("Line")
    line.set_parameters(
        line_resistance,
        line_inductance,
        line_capacitance,
        line_conductance,
    )

    load = ph3.RXLoad("Load")
    load.set_parameters(
        load_active_power,
        three_phase_power(reactive_power),
        nominal_voltage,
        False,  # parallel resistance and reactance branches
    )

    source.connect([source_node])
    line.connect([source_node, load_node])
    load.connect([load_node])

    system = SystemTopology(
        frequency,
        [source_node, load_node],
        [source, line, load],
    )

    simulation = configure_and_run(
        system,
        f"EMT_Ph3_Composite_StateSpaceExtraction_Composite_{case_name}",
    )
    return get_extraction_results(simulation)

## Explicit equivalent model

In [ ]:
def run_explicit_model(reactive_power, case_name):
    source_node = emt.SimNode("n1", PhaseType.ABC)
    load_node = emt.SimNode("n2", PhaseType.ABC)
    line_internal_node = emt.SimNode("Line_internal", PhaseType.ABC)

    source = ph3.NetworkInjection("Slack")
    source.set_parameters(
        dpsimpy.Math.single_phase_variable_to_three_phase(nominal_voltage),
        frequency,
    )

    # Explicit PiLine series branch.
    line_resistor = ph3.Resistor("Line_res")
    line_resistor.set_parameters(line_resistance)

    line_inductor = ph3.Inductor("Line_ind")
    line_inductor.set_parameters(line_inductance)

    # PiLine divides its total shunt capacitance and conductance equally
    # between the two terminals.
    line_shunt_resistance = 2.0 * np.linalg.inv(line_conductance)
    half_line_capacitance = line_capacitance / 2.0

    line_conductance_0 = ph3.Resistor("Line_con0")
    line_conductance_0.set_parameters(line_shunt_resistance)

    line_conductance_1 = ph3.Resistor("Line_con1")
    line_conductance_1.set_parameters(line_shunt_resistance)

    line_capacitor_0 = ph3.Capacitor("Line_cap0")
    line_capacitor_0.set_parameters(half_line_capacitance)

    line_capacitor_1 = ph3.Capacitor("Line_cap1")
    line_capacitor_1.set_parameters(half_line_capacitance)

    # Explicit equivalent of the parallel RXLoad.
    phase_voltage = nominal_voltage / np.sqrt(3.0)

    load_resistance = phase_voltage**2 * np.linalg.inv(load_active_power)

    load_resistor = ph3.Resistor("Load_res")
    load_resistor.set_parameters(load_resistance)

    reactive_component = None

    if reactive_power != 0.0:
        load_reactive_power = three_phase_power(reactive_power)
        load_reactance = phase_voltage**2 * np.linalg.inv(load_reactive_power)

        if reactive_power > 0.0:
            load_inductance = load_reactance / omega
            reactive_component = ph3.Inductor("Load_ind")
            reactive_component.set_parameters(load_inductance)
        else:
            load_capacitance = -np.linalg.inv(load_reactance) / omega
            reactive_component = ph3.Capacitor("Load_cap")
            reactive_component.set_parameters(load_capacitance)

    source.connect([source_node])

    line_resistor.connect([source_node, line_internal_node])
    line_inductor.connect([line_internal_node, load_node])

    line_conductance_0.connect([emt.SimNode.gnd, source_node])
    line_conductance_1.connect([emt.SimNode.gnd, load_node])
    line_capacitor_0.connect([emt.SimNode.gnd, source_node])
    line_capacitor_1.connect([emt.SimNode.gnd, load_node])

    load_resistor.connect([emt.SimNode.gnd, load_node])
    if reactive_component is not None:
        reactive_component.connect([emt.SimNode.gnd, load_node])

    # Match the internal PiLine/RXLoad subcomponent ordering so that
    # the extracted state ordering is identical in both models.
    components = [
        source,
        line_resistor,
        line_inductor,
        line_conductance_0,
        line_conductance_1,
        line_capacitor_0,
        line_capacitor_1,
        load_resistor,
    ]

    if reactive_component is not None:
        components.append(reactive_component)

    system = SystemTopology(
        frequency,
        [source_node, load_node, line_internal_node],
        components,
    )

    simulation = configure_and_run(
        system,
        f"EMT_Ph3_Composite_StateSpaceExtraction_Explicit_{case_name}",
    )
    return get_extraction_results(simulation)

## Assertions and output

In [ ]:
def validate_case(case_name, reactive_power):
    composite = run_composite_model(reactive_power, case_name)
    explicit = run_explicit_model(reactive_power, case_name)

    expected_names = expected_state_names(reactive_power)
    expected_state_count = len(expected_names)

    assert composite["state_count"] == expected_state_count, (
        f"{case_name}: composite model expected {expected_state_count} "
        f"states, got {composite['state_count']}"
    )
    assert explicit["state_count"] == expected_state_count, (
        f"{case_name}: explicit model expected {expected_state_count} "
        f"states, got {explicit['state_count']}"
    )

    assert composite["state_names"] == expected_names, (
        f"{case_name}: unexpected composite state ordering:\n"
        f"{composite['state_names']}"
    )
    assert explicit["state_names"] == expected_names, (
        f"{case_name}: unexpected explicit state ordering:\n"
        f"{explicit['state_names']}"
    )

    assert composite["Ad"].shape == explicit["Ad"].shape

    difference = composite["Ad"] - explicit["Ad"]
    maximum_absolute_difference = np.max(np.abs(difference)) if difference.size else 0.0

    reference_scale = max(
        1.0,
        np.max(np.abs(explicit["Ad"])) if explicit["Ad"].size else 0.0,
    )
    relative_difference = maximum_absolute_difference / reference_scale

    np.testing.assert_allclose(
        composite["Ad"],
        explicit["Ad"],
        rtol=0.0,
        atol=matrix_tolerance,
        err_msg=f"{case_name}: state matrices differ",
    )

    np.testing.assert_allclose(
        sort_complex(composite["z"]),
        sort_complex(explicit["z"]),
        rtol=0.0,
        atol=eigenvalue_tolerance,
        err_msg=f"{case_name}: discrete eigenvalues differ",
    )

    print("\n" + "=" * 60)
    print(f"RXLoad case: {case_name} (Q = {reactive_power:.6g} var)")
    print("=" * 60)
    print(f"Composite state count: {composite['state_count']}")
    print(f"Explicit state count:  {explicit['state_count']}")
    print("Maximum absolute Ad difference: " f"{maximum_absolute_difference:.12e}")
    print(f"Relative Ad difference:         {relative_difference:.12e}")

    print("\nExtraction states:")
    for index, state_name in enumerate(composite["state_names"]):
        print(f"  [{index}] {state_name}")

    print("\nComposite discrete-time eigenvalues:")
    for index, eigenvalue in enumerate(composite["z"]):
        print(f"  [{index}] {eigenvalue}")

    return {
        "composite": composite,
        "explicit": explicit,
        "maximum_absolute_difference": maximum_absolute_difference,
        "relative_difference": relative_difference,
    }


results = {
    case_name: validate_case(case_name, reactive_power)
    for case_name, reactive_power in load_cases.items()
}

print("\nAll composite-component state-space extraction checks passed.")